# PostgreSQL: Joins und Aggregationen

**ZHAW Data Analytics HS2026 · Gruppenprojekt «Strompreise der Gemeinden» · Teil Metehan (Bonus 2)**

Dieses Notebook überträgt die SQLite-Datenbank aus `scripts/02_sqlite_metehan.ipynb` nach PostgreSQL und wertet sie dort mit Joins und Aggregationen aus.

**Warum zusätzlich PostgreSQL?** SQLite ist eine einzelne Datei ohne Server. PostgreSQL ist ein echter Datenbankserver, wie er in Firmen läuft, und kann mehr:

| | SQLite | PostgreSQL |
| --- | --- | --- |
| Betrieb | Datei, kein Server | Server, mehrere Nutzer gleichzeitig |
| Preise speichern | `REAL` (Gleitkomma, kleine Rundungsfehler) | `NUMERIC(7,3)` (exakt auf 3 Nachkommastellen) |
| Median | fehlt, musste in pandas gerechnet werden | `percentile_cont(0.5)` |
| Statistik in SQL | nur `AVG`, `MIN`, `MAX` | zusätzlich `STDDEV`, `CORR` u. a. |
| Breitformat | `MAX(CASE WHEN ...)` | `MAX(...) FILTER (WHERE ...)` |

**Voraussetzungen**

1. PostgreSQL ist installiert und läuft (Installer von postgresql.org oder Postgres.app auf dem Mac).
2. Eine leere Datenbank existiert: im Terminal `createdb -U postgres strompreise`, oder in pgAdmin über «Create Database».
3. Der Python-Treiber ist installiert: `pip install psycopg2-binary`.
4. `sql/strompreise.db` aus Notebook 02 liegt in der Repo-Wurzel.

Das Passwort steht nicht im Code und es findet keine interaktive Abfrage statt. Verbindungen werden über die Umgebungsvariablen `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` und `PGPASSWORD` konfiguriert.

In [1]:
import os
import sqlite3
from decimal import Decimal
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from pathlib import Path

def finde_wurzel(start):
    pfad = Path(start).resolve()
    for kandidat in (pfad, *pfad.parents):
        if (kandidat / "README.md").exists():
            return kandidat
    raise FileNotFoundError("Repo-Wurzel nicht gefunden")

WURZEL = finde_wurzel(Path.cwd())
SQLITE_DB = WURZEL / "sql" / "strompreise.db"
if not SQLITE_DB.exists():
    raise FileNotFoundError(f"SQLite-Datenbank fehlt: {SQLITE_DB}")

if not os.environ.get("PGPASSWORD"):
    raise RuntimeError(
        "PGPASSWORD fehlt. Setze PGPASSWORD vor dem automatisierten Notebook-Lauf; "
        "es gibt keine interaktive Passwort-Abfrage."
    )

PG = {
    "host": os.environ.get("PGHOST", "localhost"),
    "port": int(os.environ.get("PGPORT", "5432")),
    "dbname": os.environ.get("PGDATABASE", "strompreise"),
    "user": os.environ.get("PGUSER", "postgres"),
    "password": os.environ["PGPASSWORD"],
}

con = psycopg2.connect(**PG)
lite = sqlite3.connect(SQLITE_DB)   # Quelle: Datenbank aus Notebook 02


def sql(befehl):
    # SQL-Befehl ohne Ergebnis ausfuehren (CREATE, DROP, INSERT ...)
    with con.cursor() as cur:
        cur.execute(befehl)
    con.commit()


def abfrage(befehl):
    # SELECT ausfuehren und als DataFrame zurueckgeben.
    # NUMERIC kommt aus PostgreSQL als Decimal und wird fuer pandas in float umgewandelt.
    with con.cursor() as cur:
        cur.execute(befehl)
        spalten = [d[0] for d in cur.description]
        zeilen = cur.fetchall()
    df = pd.DataFrame(zeilen, columns=spalten)
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, Decimal)).any():
            df[c] = df[c].astype(float)
    return df


abfrage("SELECT version()")

,version
0,"PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit"


## 1 · Tabellen anlegen

Das Schema entspricht dem aus SQLite. Geändert haben sich nur die Datentypen: Preise sind `NUMERIC(7,3)`, also exakte Dezimalzahlen mit drei Nachkommastellen, wie sie die ElCom liefert.

`DROP ... CASCADE` löscht zuerst alte Tabellen und Views. So lässt sich das Notebook beliebig oft neu ausführen.

In [2]:
sql('''
DROP VIEW  IF EXISTS v_analyse, v_preis_h4 CASCADE;
DROP TABLE IF EXISTS kennzahl, indikator, tarif, netzbetreiber, gemeinde CASCADE;

CREATE TABLE gemeinde (
    gemeinde_nr      INTEGER PRIMARY KEY,
    name             TEXT
);

CREATE TABLE netzbetreiber (
    netzbetreiber_id INTEGER PRIMARY KEY,
    name             TEXT NOT NULL
);

CREATE TABLE tarif (
    jahr               SMALLINT     NOT NULL,
    gemeinde_nr        INTEGER      NOT NULL REFERENCES gemeinde(gemeinde_nr),
    netzbetreiber_id   INTEGER      NOT NULL REFERENCES netzbetreiber(netzbetreiber_id),
    produkt            TEXT         NOT NULL CHECK (produkt IN ('standard', 'cheapest')),
    kategorie          TEXT         NOT NULL,
    total              NUMERIC(7,3) NOT NULL,      -- Rp./kWh exkl. MWST
    energie            NUMERIC(7,3),
    netznutzung        NUMERIC(7,3),
    abgaben            NUMERIC(7,3),
    netzzuschlag       NUMERIC(7,3),
    messtarif          NUMERIC(7,3) NOT NULL DEFAULT 0,
    messtarif_chf_jahr NUMERIC(9,3),
    PRIMARY KEY (jahr, gemeinde_nr, netzbetreiber_id, produkt, kategorie)
);

CREATE TABLE indikator (
    code         TEXT PRIMARY KEY,
    bezeichnung  TEXT NOT NULL,
    einheit      TEXT,
    periode      TEXT,
    label_quelle TEXT CHECK (label_quelle IN ('geprueft', 'abgeleitet'))
);

CREATE TABLE kennzahl (
    gemeinde_nr INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    indikator   TEXT    NOT NULL REFERENCES indikator(code),
    wert        DOUBLE PRECISION,
    status      TEXT,
    PRIMARY KEY (gemeinde_nr, indikator)
);

CREATE INDEX idx_tarif_produkt_kat ON tarif (produkt, kategorie, jahr);
''')
abfrage('''
    SELECT table_name AS tabelle, COUNT(*) AS spalten
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name
    ORDER BY table_name
''')

,tabelle,spalten
0,gemeinde,2
1,indikator,5
2,kennzahl,4
3,netzbetreiber,2
4,tarif,12


## 2 · Daten aus SQLite übernehmen

Die Daten kommen direkt aus `strompreise.db`. So rechnen SQLite und PostgreSQL garantiert mit denselben Zeilen. Die Reihenfolge ist wichtig: Erst kommen `gemeinde`, `netzbetreiber` und `indikator`, weil `tarif` und `kennzahl` auf sie verweisen.

`execute_values` schickt viele Zeilen in einem Befehl. Das ist deutlich schneller als ein `INSERT` pro Zeile.

In [3]:
TABELLEN = ["gemeinde", "netzbetreiber", "indikator", "tarif", "kennzahl"]

for tabelle in TABELLEN:
    spalten = [r[1] for r in lite.execute(f"PRAGMA table_info({tabelle})")]
    zeilen = lite.execute(f"SELECT {', '.join(spalten)} FROM {tabelle}").fetchall()
    with con.cursor() as cur:
        execute_values(cur, f"INSERT INTO {tabelle} ({', '.join(spalten)}) VALUES %s", zeilen, page_size=5000)
    con.commit()
    print(f"{tabelle:<14} {len(zeilen):>8,} Zeilen übertragen")

gemeinde          2,225 Zeilen übertragen
netzbetreiber       609 Zeilen übertragen
indikator            41 Zeilen übertragen
tarif           143,837 Zeilen übertragen
kennzahl         89,052 Zeilen übertragen


**Kontrolle:** Jede Tabelle muss in PostgreSQL gleich viele Zeilen haben wie in SQLite, und die Summe der Preisbestandteile muss das Total ergeben.

In [4]:
vergleich = pd.DataFrame({
    "tabelle": TABELLEN,
    "sqlite": [lite.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in TABELLEN],
    "postgresql": [abfrage(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in TABELLEN],
})
vergleich["gleich"] = vergleich["sqlite"] == vergleich["postgresql"]
vergleich

,tabelle,sqlite,postgresql,gleich
0,gemeinde,2225,2225,True
1,netzbetreiber,609,609,True
2,indikator,41,41,True
3,tarif,143837,143837,True
4,kennzahl,89052,89052,True


In [5]:
abfrage('''
    SELECT jahr,
           COUNT(*) AS zeilen,
           COUNT(*) FILTER (
               WHERE ABS(total - (energie + netznutzung + abgaben + netzzuschlag + messtarif)) > 0.01
           ) AS summe_falsch
    FROM tarif
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,zeilen,summe_falsch
0,2021,54210,0
1,2024,48187,0
2,2027,41440,0


## 3 · Views

Die beiden Views sind dieselben wie in SQLite. Das Drehen vom Lang- ins Breitformat schreibt PostgreSQL kürzer mit `FILTER (WHERE ...)` statt `CASE WHEN`.

In [6]:
sql('''
CREATE VIEW v_preis_h4 AS
SELECT jahr,
       gemeinde_nr,
       COUNT(*)                     AS anzahl_netzbetreiber,
       AVG(total)                   AS total,
       AVG(energie)                 AS energie,
       AVG(netznutzung + messtarif) AS netznutzung_inkl_mess,
       AVG(abgaben)                 AS abgaben,
       AVG(netzzuschlag)            AS netzzuschlag
FROM tarif
WHERE produkt = 'standard' AND kategorie = 'H4'
GROUP BY jahr, gemeinde_nr;

CREATE VIEW v_analyse AS
WITH merkmale AS (
    SELECT gemeinde_nr,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_01') AS einwohner,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_03') AS dichte,
           MAX(wert) FILTER (WHERE indikator = 'Ind_04_01') AS flaeche_total,
           MAX(wert) FILTER (WHERE indikator = 'Ind_06_03') AS beschaeftigte,
           MAX(wert) FILTER (WHERE indikator = 'Ind_01_06') AS anteil_65plus
    FROM kennzahl
    GROUP BY gemeinde_nr
)
SELECT g.gemeinde_nr,
       g.name     AS gemeinde,
       p21.anzahl_netzbetreiber,
       p21.total  AS total_2021,
       p24.total  AS total_2024,
       p27.total  AS total_2027,
       m.einwohner, m.dichte, m.flaeche_total, m.beschaeftigte, m.anteil_65plus
FROM gemeinde g
JOIN v_preis_h4 p21 ON p21.gemeinde_nr = g.gemeinde_nr AND p21.jahr = 2021
JOIN v_preis_h4 p24 ON p24.gemeinde_nr = g.gemeinde_nr AND p24.jahr = 2024
JOIN v_preis_h4 p27 ON p27.gemeinde_nr = g.gemeinde_nr AND p27.jahr = 2027
JOIN merkmale  m   ON m.gemeinde_nr   = g.gemeinde_nr;
''')
abfrage("SELECT COUNT(*) AS gemeinden FROM v_analyse")

,gemeinden
0,2087


## 4 · Auswertungen mit Joins und Aggregationen

**4.1 · Verteilung der H4-Preise pro Jahr.** Median und Standardabweichung rechnet PostgreSQL direkt. In SQLite brauchte der Median noch pandas.

In [7]:
abfrage('''
    SELECT jahr,
           COUNT(*)                                                   AS gemeinden,
           ROUND(AVG(total), 2)                                       AS mittelwert,
           ROUND(percentile_cont(0.5) WITHIN GROUP (ORDER BY total)::numeric, 2) AS median,
           ROUND(STDDEV(total), 2)                                    AS std_abw,
           ROUND(MIN(total), 2)                                       AS minimum,
           ROUND(MAX(total), 2)                                       AS maximum
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,gemeinden,mittelwert,median,std_abw,minimum,maximum
0,2021,2201,20.86,20.63,3.00,7.96,27.65
1,2024,2140,32.00,32.15,3.73,10.22,50.62
2,2027,2109,27.24,26.91,3.06,10.56,45.88


**4.2 · Anteil der Bestandteile am Total.** Der Anteil zeigt, welcher Teil des Preises den Anstieg getragen hat. Die Spalte `netznutzung_inkl_mess` macht die Jahre vergleichbar, weil der Messtarif ab 2026 separat ausgewiesen wird.

In [8]:
abfrage('''
    SELECT jahr,
           ROUND(AVG(total), 2)                                      AS total,
           ROUND(AVG(energie), 2)                                    AS energie,
           ROUND(AVG(netznutzung_inkl_mess), 2)                      AS netznutzung,
           ROUND(AVG(abgaben + netzzuschlag), 2)                     AS abgaben_zuschlag,
           ROUND(100 * SUM(energie) / SUM(total), 1)                 AS anteil_energie_pct,
           ROUND(100 * SUM(netznutzung_inkl_mess) / SUM(total), 1)   AS anteil_netz_pct
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''')

,jahr,total,energie,netznutzung,abgaben_zuschlag,anteil_energie_pct,anteil_netz_pct
0,2021,20.86,7.95,9.76,3.14,38.1,46.8
1,2024,32.00,15.59,13.22,3.18,48.7,41.3
2,2027,27.24,11.19,12.86,3.20,41.1,47.2


**4.3 · Die teuersten grösseren Netzbetreiber 2027.** Hier werden `tarif` und `netzbetreiber` verbunden, pro Betreiber aggregiert und mit `HAVING` auf Betreiber mit mindestens 10 Gemeinden eingeschränkt. `RANK()` ist eine Fensterfunktion, sie nummeriert die Zeilen, ohne sie zusammenzufassen.

In [9]:
abfrage('''
    SELECT RANK() OVER (ORDER BY AVG(t.total) DESC) AS rang,
           n.name                                   AS netzbetreiber,
           COUNT(DISTINCT t.gemeinde_nr)            AS gemeinden,
           ROUND(AVG(t.total), 2)                   AS preis_2027
    FROM tarif t
    JOIN netzbetreiber n ON n.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    GROUP BY n.netzbetreiber_id, n.name
    HAVING COUNT(DISTINCT t.gemeinde_nr) >= 10
    ORDER BY rang
    LIMIT 10
''')

,rang,netzbetreiber,gemeinden,preis_2027
0,1,Primeo Netz AG,64,31.33
1,2,Elektrizitätswerk des Kantons Schaffhausen AG,26,30.82
2,3,Repower AG,33,29.88
3,4,AEW Energie AG,75,29.75
4,5,St. Gallisch-Appenzellische Kraftwerke AG,40,29.45
5,6,Società Elettrica Sopracenerina SA (Ticino),36,28.80
6,7,"Genossenschaft Elektra, Jegenstorf",22,28.78
7,8,Romande Energie SA,262,28.36
8,9,Société des Forces Électriques de La Goule SA,16,28.04
9,10,EBL (Genossenschaft Elektra Baselland),50,27.94


**4.4 · Geht der Preis überall wieder zurück?** Die Fensterfunktion `LAG` holt pro Gemeinde den Preis der vorherigen Periode. Danach wird gezählt, wo der Preis 2027 gesunken ist und wo er noch über dem Niveau von 2021 liegt. Das ist eine erste Antwort auf die Frage aus dem Projektplan, «ob es nicht überall wieder zurückgeht».

In [10]:
abfrage('''
    WITH verlauf AS (
        SELECT gemeinde_nr, jahr, total,
               LAG(total) OVER (PARTITION BY gemeinde_nr ORDER BY jahr) AS vorher,
               FIRST_VALUE(total) OVER (PARTITION BY gemeinde_nr ORDER BY jahr) AS stand_2021
        FROM v_preis_h4
        WHERE gemeinde_nr IN (SELECT gemeinde_nr FROM v_analyse)   -- nur Gemeinden mit allen 3 Jahren
    )
    SELECT COUNT(*)                                              AS gemeinden,
           COUNT(*) FILTER (WHERE total < vorher)                AS guenstiger_als_2024,
           COUNT(*) FILTER (WHERE total >= vorher)               AS gleich_oder_teurer_als_2024,
           COUNT(*) FILTER (WHERE total <= stand_2021)           AS zurueck_auf_2021,
           ROUND(AVG(total - vorher), 2)                         AS mittl_aenderung_24_27,
           ROUND(AVG(total - stand_2021), 2)                     AS mittl_aenderung_21_27
    FROM verlauf
    WHERE jahr = 2027
''')

,gemeinden,guenstiger_als_2024,gleich_oder_teurer_als_2024,zurueck_auf_2021,mittl_aenderung_24_27,mittl_aenderung_21_27
0,2087,1922,165,227,-4.74,6.39


**4.5 · Zusammenhang mit Gemeindemerkmalen.** `CORR` berechnet den Korrelationskoeffizienten nach Pearson direkt in SQL. Dafür werden Preise und Kennzahlen verbunden. Einwohner und Dichte werden logarithmiert (`LN`), weil wenige grosse Städte die Werte sonst dominieren.

Das ist nur eine erste Übersicht ohne p-Werte. Die Tests mit p-Werten macht Subraj in der Analyse (Bonus 4).

In [11]:
abfrage('''
    SELECT p.jahr,
           COUNT(*)                                      AS gemeinden,
           ROUND(CORR(p.total, LN(m.einwohner))::numeric, 3)  AS r_ln_einwohner,
           ROUND(CORR(p.total, LN(m.dichte))::numeric, 3)     AS r_ln_dichte,
           ROUND(CORR(p.total, m.anteil_65plus)::numeric, 3)  AS r_anteil_65plus,
           ROUND(CORR(p.total, p.anzahl_netzbetreiber)::numeric, 3) AS r_anzahl_betreiber
    FROM v_preis_h4 p
    JOIN v_analyse m ON m.gemeinde_nr = p.gemeinde_nr
    GROUP BY p.jahr
    ORDER BY p.jahr
''')

,jahr,gemeinden,r_ln_einwohner,r_ln_dichte,r_anteil_65plus,r_anzahl_betreiber
0,2021,2087,-0.145,-0.134,0.216,-0.053
1,2024,2087,0.115,0.205,-0.106,0.078
2,2027,2087,0.031,0.070,-0.064,0.093


**4.6 · Preise nach Einwohner-Quartil.** `NTILE(4)` teilt die Gemeinden in vier gleich grosse Gruppen nach Einwohnerzahl. Mit `FILTER` steht dann jedes Jahr in einer eigenen Spalte.

In [12]:
abfrage('''
    WITH quartile AS (
        SELECT gemeinde_nr, NTILE(4) OVER (ORDER BY einwohner) AS quartil
        FROM v_analyse
    )
    SELECT q.quartil,
           COUNT(DISTINCT q.gemeinde_nr)                         AS gemeinden,
           MIN(a.einwohner)::int                                 AS einwohner_von,
           MAX(a.einwohner)::int                                 AS einwohner_bis,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2021), 2)   AS preis_2021,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2024), 2)   AS preis_2024,
           ROUND(AVG(p.total) FILTER (WHERE p.jahr = 2027), 2)   AS preis_2027
    FROM quartile q
    JOIN v_analyse  a ON a.gemeinde_nr = q.gemeinde_nr
    JOIN v_preis_h4 p ON p.gemeinde_nr = q.gemeinde_nr
    GROUP BY q.quartil
    ORDER BY q.quartil
''')

,quartil,gemeinden,einwohner_von,einwohner_bis,preis_2021,preis_2024,preis_2027
0,1,522,32,764,21.62,31.36,26.99
1,2,522,769,1627,20.86,32.04,27.41
2,3,522,1629,3924,20.69,31.96,27.45
3,4,521,3932,420217,20.28,32.59,27.16


## 5 · Kontrolle gegen SQLite

Die Analysetabelle aus PostgreSQL muss dieselben Gemeinden und Werte liefern wie die aus SQLite. Kleine Unterschiede in der letzten Stelle sind möglich, weil SQLite mit Gleitkommazahlen rechnet und PostgreSQL exakt.

In [13]:
aus_pg = abfrage("SELECT * FROM v_analyse").set_index("gemeinde_nr").sort_index()
aus_lite = pd.read_sql("SELECT * FROM v_analyse", lite).set_index("gemeinde_nr").sort_index()

print("Gemeinden PostgreSQL / SQLite:", len(aus_pg), "/", len(aus_lite))
print("Gleiche Gemeindenummern:     ", aus_pg.index.equals(aus_lite.index))
for spalte in ["total_2021", "total_2024", "total_2027", "einwohner", "dichte", "beschaeftigte"]:
    diff = (aus_pg[spalte].astype(float) - aus_lite[spalte]).abs().max()
    print(f"{spalte:<15} grösste Abweichung: {diff:.2e}")

Gemeinden PostgreSQL / SQLite: 2087 / 2087
Gleiche Gemeindenummern:      True
total_2021      grösste Abweichung: 3.55e-15
total_2024      grösste Abweichung: 7.11e-15
total_2027      grösste Abweichung: 7.11e-15
einwohner       grösste Abweichung: 0.00e+00
dichte          grösste Abweichung: 0.00e+00
beschaeftigte   grösste Abweichung: 0.00e+00


## 6 · Fazit

- Beide Datenbanken enthalten dieselben Daten und liefern dieselbe Analysetabelle. Die Kontrollen oben zeigen das.
- PostgreSQL lohnt sich vor allem für Auswertungen: Median, Standardabweichung, Korrelation und Fensterfunktionen wie `LAG`, `RANK` und `NTILE` laufen direkt in SQL.
- Für die Gruppe ändert sich nichts. Subraj kann weiter mit `v_analyse` aus SQLite arbeiten, die Zahlen sind identisch.

In [14]:
con.close()
lite.close()